In [20]:
import os, json, re
from glob import glob
from dataclasses import dataclass, asdict
from typing import List, Dict

import openai
import pandas as pd
import evaluate  

!pip install openai rouge-score pandas
import os
import json
import pandas as pd
import openai
from rouge_score import rouge_scorer
from dotenv import load_dotenv


[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# %% [markdown]
# # Build B-TEXT, A-TAG & A-ONLY Datasets

# %% [code]
import json
import pandas as pd

# Path to your scraped+annotated JSON
json_path = "/Users/gon/Documents/Spring 2025/H2AI/v2/doc_parsing/layoutlm/output_503/agent_workflow_memory_processed.json"

# Load JSON
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f) 

# Support both a single dict or a list of docs
if isinstance(data, dict) and "document" in data:
    docs = [data]
elif isinstance(data, list):
    docs = data
else:
    raise ValueError("Unexpected JSON structure: must be a dict with 'document' or a list of such dicts")

# %% [markdown]
# ## Iterate and assemble each dataset

# %% [code]
b_text_records = []
a_tag_records  = []
a_only_records = []

for doc in docs:
    doc_id      = doc["document"]["id"]
    paragraphs  = doc["paragraphs"]
    annotations = doc["annotations"]

    # 1) B-TEXT: concatenate all paragraphs in reading order
    paras_sorted = sorted(paragraphs, key=lambda p: p["character_start"])
    full_text    = "\n".join(p["text"] for p in paras_sorted)

    # 2) A-TAG: inject tags around each annotated span
    #    group annotations by paragraph for faster lookup
    ann_by_para = {}
    for ann in annotations:
        ann_by_para.setdefault(ann["paragraph_id"], []).append(ann)

    tagged_paras = []
    for p in paras_sorted:
        text = p["text"]
        anns = ann_by_para.get(p["id"], [])
        # sort descending so earlier insertions don't shift later offsets
        anns_sorted = sorted(anns, key=lambda a: a["referenced_char_start"], reverse=True)

        for a in anns_sorted:
            # map doc-level offsets to paragraph-level
            start = a["referenced_char_start"] - p["character_start"]
            end   = a["referenced_char_end"]   - p["character_start"]

            if a["type"] == "highlight":
                ot, ct = "<hl>", "</hl>"
            elif a["type"] == "comment":
                ot, ct = "<c>",  "</c>"
            elif a["type"] == "symbol":
                ot, ct = "<s>",  "</s>"
            else:
                continue

            # inject
            text = text[:start] + ot + text[start:end] + ct + text[end:]

        tagged_paras.append(text)

    tagged_text = "\n".join(tagged_paras)

    # 3) A-ONLY: grab the user-supplied annotated_text in order of appearance
    anns_sorted_all = sorted(annotations, key=lambda a: a["referenced_char_start"])
    only_texts      = [a["annotated_text"] for a in anns_sorted_all]
    a_only_text     = " ".join(only_texts)

    # register our examples
    b_text_records.append({"doc_id": doc_id, "text": full_text})
    a_tag_records .append({"doc_id": doc_id, "text": tagged_text})
    a_only_records.append({"doc_id": doc_id, "annotations": a_only_text})

# %% [markdown]
# ## Build DataFrames & save

# %% [code]
df_b_text = pd.DataFrame(b_text_records)
df_a_tag  = pd.DataFrame(a_tag_records)
df_a_only = pd.DataFrame(a_only_records)

# Save out for your training scripts
df_b_text.to_csv("data_processed/B_TEXT.csv", index=False)
df_a_tag .to_csv("data_processed/A_TAG.csv",  index=False)
df_a_only.to_csv("data_processed/A_ONLY.csv", index=False)

print("✅ Datasets built and saved: B_TEXT.csv, A_TAG.csv, A_ONLY.csv")

✅ Datasets built and saved: B_TEXT.csv, A_TAG.csv, A_ONLY.csv


In [21]:
JSON_PATH = "/Users/gon/Documents/Spring 2025/H2AI/v2/doc_parsing/layoutlm/output_503/agent_workflow_memory_processed.json"

load_dotenv()

# Load your API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Load the original JSON
with open(JSON_PATH, "r", encoding="utf-8") as f:
    orig = json.load(f)

# Support either a single doc or a list of docs
docs = [orig] if isinstance(orig, dict) and "document" in orig else orig

# Load the three CSVs
df_b  = pd.read_csv("data_processed/B_TEXT.csv")
df_at = pd.read_csv("data_processed/A_TAG.csv")
df_ao = pd.read_csv("data_processed/A_ONLY.csv")

# Map doc_id → ground-truth summary
orig_summary = {
    d["document"]["id"]: d["document"]["summary"]
    for d in docs
}

# Define GPT-4o summarization function

def generate_summary(text: str, mode: str) -> str:
    """
    mode == "B-TEXT" : summarize raw text
    mode == "A-TAG"  : summarize with <hl>/<s>/<c> annotation tags
    mode == "A-ONLY": summarize using only the concatenated annotations
    """
    if mode == "B-TEXT":
        system_msg = "You are an AI tutor."
        user_msg = (
            "Provide a concise, accurate summary of the following academic text:\n\n"
            f"{text}"
        )
    elif mode == "A-TAG":
        system_msg = "You are an AI tutor helping a student interpret annotations."
        user_msg = (
            "Generate a personalized summary that leverages the student's annotations, "
            "explaining highlights (<hl>…</hl>), symbols (<s>…</s>), and comments (<c>…</c>):\n\n"
            f"{text}"
        )
    elif mode == "A-ONLY":
        system_msg = "You are an AI tutor."
        user_msg = (
            "Using only the student's annotations with tags (<hl>, <s>, <c>), craft a personalized summary "
            "that explains the key concepts highlighted by those annotations:\n\n"
            f"{text}"
        )
    else:
        raise ValueError(f"Unknown mode: {mode}")



    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg}
        ],
        temperature=0.3
    )
    return resp.choices[0].message.content.strip()

# Generate all summaries

records = []
for doc_id in df_b["doc_id"].unique():
    txt_b  = df_b.loc[df_b.doc_id==doc_id, "text"].iloc[0]
    txt_at = df_at.loc[df_at.doc_id==doc_id, "text"].iloc[0]
    ann_o  = df_ao.loc[df_ao.doc_id==doc_id, "annotations"].iloc[0]
    
    s_b  = generate_summary(txt_b,  "B-TEXT")
    s_at = generate_summary(txt_at, "A-TAG")
    s_ao = generate_summary(ann_o,  "A-ONLY")
    
    records += [
        {"doc_id": doc_id, "mode": "B-TEXT",  "generated": s_b},
        {"doc_id": doc_id, "mode": "A-TAG",   "generated": s_at},
        {"doc_id": doc_id, "mode": "A-ONLY",  "generated": s_ao},
    ]

df_gen = pd.DataFrame(records)

In [15]:
def display_dataframe_to_user(title, df):
    print(f"\n{title}")
    return df 

# Evaluate with ROUGE (Recall)

scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
metrics = []

for _, row in df_gen.iterrows():
    ref = orig_summary[row["doc_id"]]
    hyp = row["generated"]
    sc  = scorer.score(ref, hyp)
    metrics.append({
        "doc_id":         row["doc_id"],
        "mode":           row["mode"],
        "rouge1_recall":  sc["rouge1"].recall,
        "rouge2_recall":  sc["rouge2"].recall,
        "rougeL_recall":  sc["rougeL"].recall,
    })

df_metrics = pd.DataFrame(metrics)

# Show Comparison Table

display_dataframe_to_user("Summary Comparison Table", df_metrics)
print(df_metrics)


Summary Comparison Table
         doc_id    mode  rouge1_recall  rouge2_recall  rougeL_recall
0  doc_82805800  B-TEXT       0.028040       0.010037       0.015791
1  doc_82805800   A-TAG       0.037043       0.017565       0.020809
2  doc_82805800  A-ONLY       0.026712       0.015793       0.016529
